Purpose
---
Prepare European port datasets and consolidate them into a single CSV.

Pipeline:
1. Filter WPI/Ocean Data Platform ports to European countries.
2. Filter CNSP ports to EU + associated overseas territories.
3. Consolidate WPI and CNSP ports into a single CSV.

Inputs:
    ocean_data_platform_ports.csv
    cnsp_ports.csv

Intermediate outputs:
    wpi_european_ports.csv
    cnsp_ports_ue_filtered.csv

Final output:
    ports_consolidated.csv

# Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/BCKG/CSV Ports"

Mounted at /content/drive
/content/drive/MyDrive/BCKG/CSV Ports


In [ ]:
import re
import pandas as pd

WPI_INPUT_FILE = "ocean_data_platform_ports.csv"
WPI_FILTERED_FILE = "wpi_european_ports.csv"

CNSP_INPUT_FILE = "cnsp_ports.csv"
CNSP_FILTERED_FILE = "cnsp_ports_ue_filtered.csv"

CONSOLIDATED_OUTPUT_FILE = "ports_consolidated.csv"

CNSP_SOURCE_URL = "https://www.data.gouv.fr/api/1/datasets/r/60fe965d-5888-493b-9321-24bc3b1f84db"
WPI_SOURCE = "WPI"

# Country names as represented in the WPI / Ocean Data Platform file.
WPI_EUROPEAN_COUNTRIES = {
    'France', 'Spain', 'Italy', 'Portugal', 'Germany', 'Belgium', 'Netherlands',
    'United Kingdom', 'Ireland', 'Denmark', 'Sweden', 'Norway', 'Finland',
    'Poland', 'Lithuania', 'Latvia', 'Estonia', 'Iceland', 'Malta', 'Cyprus',
    'Croatia', 'Slovenia', 'Greece', 'Monaco', 'Albania', 'Montenegro',
    'Bosnia and Herzegovina', 'Romania', 'Bulgaria', 'Ukraine', 'Russia',
    'Turkey', 'Georgia'
}

# EU countries as ISO2 codes in the CNSP file.
EU_COUNTRIES = {
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR",
    "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL",
    "PL", "PT", "RO", "SK", "SI", "ES", "SE"
}

# Overseas territories linked to EU countries.
EU_OVERSEAS_TERRITORIES = {
    "GF",  # French Guiana
    "GP",  # Guadeloupe
    "MQ",  # Martinique
    "RE",  # Réunion
    "YT",  # Mayotte
    "PM",  # Saint Pierre and Miquelon
    "BL",  # Saint Barthélemy
    "MF",  # Saint Martin, French part
    "NC",  # New Caledonia
    "PF",  # French Polynesia
    "WF",  # Wallis and Futuna
    "TF",  # French Southern Territories
    "AW",  # Aruba
    "CW",  # Curaçao
    "SX",  # Sint Maarten
    "BQ",  # Caribbean Netherlands
    "GL",  # Greenland
    "FO"   # Faroe Islands
}

VALID_CNSP_CODES = EU_COUNTRIES | EU_OVERSEAS_TERRITORIES

INVALID_VALUES = {
    "",
    "PORT_NAME",
    "LOCODE",
    "MAIN PORT NAME",
    "UN/LOCODE",
    "PUBLICATION LINK",
    "COUNTRY CODE",
    "REGION NAME",
    "POSITION",
    "LATITUDE",
    "LONGITUDE",
    "SOURCE",
    "FAO_AREAS",
}

# Helpers

In [ ]:
def as_clean_string(value):
    """Return a stripped string, or an empty string for missing values."""
    if pd.isna(value):
        return ""
    return str(value).strip()


def is_invalid_value(value):
    """Check whether a value is empty or looks like a header accidentally parsed as data."""
    text = as_clean_string(value).upper()
    return text in INVALID_VALUES


def clean_port_name(value):
    """Clean a port name and discard invalid/header-like values."""
    text = as_clean_string(value)
    if is_invalid_value(text):
        return ""
    return text


def clean_locode(value):
    """Clean a UN/LOCODE value and discard invalid/header-like values."""
    text = as_clean_string(value).upper()
    if is_invalid_value(text):
        return ""
    return text


def normalize_text(value):
    """Normalize text for matching purposes while preserving display values elsewhere."""
    text = clean_port_name(value).upper()
    if not text:
        return ""
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def make_coordinates(lat, lon):
    """Build a lat,lon,precision coordinate string from latitude and longitude columns."""
    if pd.isna(lat) or pd.isna(lon):
        return ""
    lat = str(lat).strip()
    lon = str(lon).strip()
    if not lat or not lon:
        return ""
    return f"{lat},{lon},0.000001"


def geometry_to_position(value):
    """Convert a WKT POINT(lon lat) geometry into lat,lon,precision format."""
    if pd.isna(value):
        return None

    value = str(value).strip()
    match = re.match(r'^POINT\s*\(\s*([-\d\.]+)\s+([-\d\.]+)\s*\)$', value)
    if not match:
        return None

    lon = float(match.group(1))
    lat = float(match.group(2))
    precision = 0.000001

    return f"{lat:.6f},{lon:.6f},{precision:.6f}"


def merge_sources(*values):
    """Merge source markers into a comma-separated unique list."""
    items = []
    for value in values:
        if pd.isna(value) or value is None:
            continue
        for part in str(value).split(","):
            part = part.strip()
            if part and part not in items:
                items.append(part)
    return ",".join(items)


def first_not_empty(*values):
    """Return the first non-empty value from the provided candidates."""
    for value in values:
        if pd.notna(value) and str(value).strip() != "":
            return value
    return ""


def build_country_region(country, region):
    """Build a compact country-region string for the consolidated port output."""
    country = as_clean_string(country)
    region = as_clean_string(region)

    if is_invalid_value(country):
        country = ""
    if is_invalid_value(region):
        region = ""

    parts = [x for x in [country, region] if x]
    return " - ".join(parts)


# =========================
# FAO area helpers
# =========================

def parse_fao_areas(value):
    """
    Parse FAO area lists while preserving comma-suffixed fragments.

    Examples:
        {34.3.4,34.4.1} -> ['34.3.4', '34.4.1']
        {27.3.b,c,27.3.c.33} -> ['27.3.b,c', '27.3.c.33']

    Rule:
        - split by comma
        - if a token is only letters, attach it to the previous token
    """
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text or text == "{}":
        return []

    if text.startswith("{") and text.endswith("}"):
        text = text[1:-1].strip()

    if not text:
        return []

    raw_parts = [p.strip() for p in text.split(",") if p.strip()]
    if not raw_parts:
        return []

    merged = []
    for part in raw_parts:
        if re.fullmatch(r"[A-Za-z]+", part) and merged:
            merged[-1] = f"{merged[-1]},{part}"
        else:
            merged.append(part)

    # Remove duplicates while preserving order.
    seen = set()
    result = []
    for item in merged:
        if item not in seen:
            seen.add(item)
            result.append(item)

    return result


def format_fao_areas(value):
    """Format FAO areas as repeated braced values expected by the downstream import."""
    areas = parse_fao_areas(value)
    if not areas:
        return ""
    return ",".join(f"{{{area}}}" for area in areas)

# Step 1: Filter WPI / Ocean Data Platform ports

In [ ]:
def filter_wpi_ports():
    """Filter WPI/Ocean Data Platform ports to European countries and normalize coordinates."""
    df = pd.read_csv(WPI_INPUT_FILE)

    filtered = df[df['Country Code'].isin(WPI_EUROPEAN_COUNTRIES)].copy()

    # Remove internal spaces from UN/LOCODE values.
    filtered['UN/LOCODE'] = (
        filtered['UN/LOCODE']
        .fillna('')
        .astype(str)
        .str.replace(' ', '', regex=False)
    )

    # Convert WKT geometry into the coordinate format used later by the import.
    filtered['Position'] = filtered['geometry'].apply(geometry_to_position)

    # The original WKT geometry is no longer needed once Position has been created.
    filtered = filtered.drop(columns=['geometry'])

    filtered.to_csv(
        WPI_FILTERED_FILE,
        sep=';',
        index=False,
        encoding='utf-8-sig'
    )

    print("WPI file successfully generated")
    print(f"WPI exported rows: {len(filtered)}")

# Step 2: Filter CNSP ports

In [ ]:
def filter_cnsp_ports():
    """Filter CNSP ports to EU and associated overseas territories."""
    df = pd.read_csv(CNSP_INPUT_FILE)

    # Normalize country codes before filtering.
    df["country_code_iso2"] = df["country_code_iso2"].astype(str).str.upper().str.strip()

    # Keep only EU countries and associated overseas territories.
    df = df[df["country_code_iso2"].isin(VALID_CNSP_CODES)].copy()

    # Create coordinates column with format: lat,lon,precision.
    df["coordinates"] = df.apply(
        lambda row: make_coordinates(row.get("latitude"), row.get("longitude")),
        axis=1
    )

    # Add source column for downstream references.
    df["source"] = CNSP_SOURCE_URL

    df.to_csv(CNSP_FILTERED_FILE, sep=";", index=False, encoding="utf-8")

    print(f"CNSP file generated: {CNSP_FILTERED_FILE}")
    print(f"CNSP resulting rows: {len(df)}")

# Step 3: Consolidate prepared port files

In [ ]:
def load_prepared_ports():
    """Load the filtered WPI and CNSP files produced by the previous steps."""
    cnsp = pd.read_csv(CNSP_FILTERED_FILE, sep=";")
    wpi = pd.read_csv(WPI_FILTERED_FILE, sep=";")
    return cnsp, wpi


def prepare_cnsp_for_matching(cnsp):
    """Rename and normalize CNSP columns before matching."""
    cnsp = cnsp.rename(columns={
        "port_name": "cnsp_port_name",
        "locode": "cnsp_locode",
        "region": "cnsp_region",
        "country_code_iso2": "cnsp_country",
        "fao_areas": "cnsp_fao_areas",
        "coordinates": "cnsp_coordinates",
        "source": "cnsp_source"
    })

    cnsp["cnsp_port_name"] = cnsp["cnsp_port_name"].apply(clean_port_name)
    cnsp["cnsp_locode"] = cnsp["cnsp_locode"].apply(clean_locode)
    cnsp["cnsp_locode_clean"] = cnsp["cnsp_locode"]
    cnsp["cnsp_name_clean"] = cnsp["cnsp_port_name"].apply(normalize_text)
    cnsp["cnsp_fao_areas"] = cnsp["cnsp_fao_areas"].apply(format_fao_areas)

    if "latitude" in cnsp.columns and "longitude" in cnsp.columns:
        missing = cnsp["cnsp_coordinates"].isna() | (cnsp["cnsp_coordinates"].astype(str).str.strip() == "")
        cnsp.loc[missing, "cnsp_coordinates"] = cnsp.loc[missing].apply(
            lambda row: make_coordinates(row.get("latitude"), row.get("longitude")),
            axis=1
        )

    cnsp["cnsp_country_region"] = cnsp.apply(
        lambda row: build_country_region(row.get("cnsp_country"), row.get("cnsp_region")),
        axis=1
    )

    # Keep rows with at least one usable identifying value.
    cnsp = cnsp[
        (cnsp["cnsp_port_name"].astype(str).str.strip() != "") |
        (cnsp["cnsp_locode_clean"].astype(str).str.strip() != "")
    ].copy()

    return cnsp


def prepare_wpi_for_matching(wpi):
    """Rename and normalize WPI columns before matching."""
    wpi = wpi.rename(columns={
        "Main Port Name": "wpi_port_name",
        "UN/LOCODE": "wpi_locode",
        "Country Code": "wpi_country",
        "Region Name": "wpi_region",
        "Latitude": "wpi_latitude",
        "Longitude": "wpi_longitude",
        "Position": "wpi_coordinates"
    })

    wpi["wpi_port_name"] = wpi["wpi_port_name"].apply(clean_port_name)
    wpi["wpi_locode"] = wpi["wpi_locode"].apply(clean_locode)
    wpi["wpi_locode_clean"] = wpi["wpi_locode"]
    wpi["wpi_name_clean"] = wpi["wpi_port_name"].apply(normalize_text)
    wpi["wpi_source"] = WPI_SOURCE

    missing = wpi["wpi_coordinates"].isna() | (wpi["wpi_coordinates"].astype(str).str.strip() == "")
    wpi.loc[missing, "wpi_coordinates"] = wpi.loc[missing].apply(
        lambda row: make_coordinates(row.get("wpi_latitude"), row.get("wpi_longitude")),
        axis=1
    )

    wpi["wpi_country_region"] = wpi.apply(
        lambda row: build_country_region(row.get("wpi_country"), row.get("wpi_region")),
        axis=1
    )

    # Keep rows with at least one usable identifying value.
    wpi = wpi[
        (wpi["wpi_port_name"].astype(str).str.strip() != "") |
        (wpi["wpi_locode_clean"].astype(str).str.strip() != "")
    ].copy()

    return wpi


def match_ports(cnsp, wpi):
    """
    Match CNSP and WPI ports in two passes.

    First pass:
        exact match by LOCODE.

    Second pass:
        remaining records matched by normalized port name.
    """
    # First pass: match by LOCODE.
    cnsp_by_locode = cnsp[cnsp["cnsp_locode_clean"] != ""].copy()
    wpi_by_locode = wpi[wpi["wpi_locode_clean"] != ""].copy()

    matched_locode = pd.merge(
        cnsp_by_locode,
        wpi_by_locode,
        left_on="cnsp_locode_clean",
        right_on="wpi_locode_clean",
        how="inner"
    )

    matched_cnsp_locodes = set(matched_locode["cnsp_locode_clean"].dropna())
    matched_wpi_locodes = set(matched_locode["wpi_locode_clean"].dropna())

    cnsp_left = cnsp[~cnsp["cnsp_locode_clean"].isin(matched_cnsp_locodes)].copy()
    wpi_left = wpi[~wpi["wpi_locode_clean"].isin(matched_wpi_locodes)].copy()

    # Second pass: match remaining rows by normalized name.
    cnsp_by_name = cnsp_left[cnsp_left["cnsp_name_clean"] != ""].copy()
    wpi_by_name = wpi_left[wpi_left["wpi_name_clean"] != ""].copy()

    matched_name = pd.merge(
        cnsp_by_name,
        wpi_by_name,
        left_on="cnsp_name_clean",
        right_on="wpi_name_clean",
        how="inner"
    )

    matched_cnsp_names = set(matched_name["cnsp_name_clean"].dropna())
    matched_wpi_names = set(matched_name["wpi_name_clean"].dropna())

    cnsp_only = cnsp_left[~cnsp_left["cnsp_name_clean"].isin(matched_cnsp_names)].copy()
    wpi_only = wpi_left[~wpi_left["wpi_name_clean"].isin(matched_wpi_names)].copy()

    return matched_locode, matched_name, cnsp_only, wpi_only


def build_output(df):
    """Build final-schema rows from matched CNSP/WPI records."""
    rows = []

    for _, row in df.iterrows():
        port_name = first_not_empty(row.get("wpi_port_name"), row.get("cnsp_port_name"))
        locode = first_not_empty(row.get("wpi_locode"), row.get("cnsp_locode"))
        coordinates = first_not_empty(row.get("wpi_coordinates"), row.get("cnsp_coordinates"))
        country_region = first_not_empty(row.get("wpi_country_region"), row.get("cnsp_country_region"))
        fao_areas = first_not_empty(row.get("cnsp_fao_areas"))
        sources = merge_sources(row.get("cnsp_source"), row.get("wpi_source"))

        port_name = clean_port_name(port_name)
        locode = clean_locode(locode)

        if port_name or locode:
            rows.append({
                "sources": sources,
                "port_name": port_name,
                "coordinates": coordinates,
                "locode": locode,
                "country_region": country_region,
                "fao_areas": fao_areas
            })

    return pd.DataFrame(rows)


def build_cnsp_only_output(cnsp_only):
    """Build final-schema rows for unmatched CNSP records."""
    return pd.DataFrame([{
        "sources": row.get("cnsp_source", ""),
        "port_name": clean_port_name(row.get("cnsp_port_name", "")),
        "coordinates": row.get("cnsp_coordinates", ""),
        "locode": clean_locode(row.get("cnsp_locode", "")),
        "country_region": row.get("cnsp_country_region", ""),
        "fao_areas": row.get("cnsp_fao_areas", "")
    } for _, row in cnsp_only.iterrows()
      if clean_port_name(row.get("cnsp_port_name", "")) or clean_locode(row.get("cnsp_locode", ""))])


def build_wpi_only_output(wpi_only):
    """Build final-schema rows for unmatched WPI records."""
    return pd.DataFrame([{
        "sources": row.get("wpi_source", ""),
        "port_name": clean_port_name(row.get("wpi_port_name", "")),
        "coordinates": row.get("wpi_coordinates", ""),
        "locode": clean_locode(row.get("wpi_locode", "")),
        "country_region": row.get("wpi_country_region", ""),
        "fao_areas": ""
    } for _, row in wpi_only.iterrows()
      if clean_port_name(row.get("wpi_port_name", "")) or clean_locode(row.get("wpi_locode", ""))])


def consolidate_ports():
    """Consolidate filtered WPI and CNSP port datasets into one final dataframe."""
    cnsp, wpi = load_prepared_ports()

    cnsp = prepare_cnsp_for_matching(cnsp)
    wpi = prepare_wpi_for_matching(wpi)

    matched_locode, matched_name, cnsp_only, wpi_only = match_ports(cnsp, wpi)

    out_locode = build_output(matched_locode)
    out_name = build_output(matched_name)
    out_cnsp_only = build_cnsp_only_output(cnsp_only)
    out_wpi_only = build_wpi_only_output(wpi_only)

    final_df = pd.concat(
        [out_locode, out_name, out_cnsp_only, out_wpi_only],
        ignore_index=True
    )

    final_df["port_name"] = final_df["port_name"].apply(clean_port_name)
    final_df["locode"] = final_df["locode"].apply(clean_locode)
    final_df["fao_areas"] = final_df["fao_areas"].apply(format_fao_areas)

    final_df = final_df[
        (final_df["port_name"].astype(str).str.strip() != "") |
        (final_df["locode"].astype(str).str.strip() != "")
    ].copy()

    final_df = final_df.drop_duplicates(
        subset=["port_name", "locode", "coordinates", "country_region", "fao_areas", "sources"]
    ).reset_index(drop=True)

    final_df = final_df.sort_values(by=["port_name", "locode"], na_position="last").reset_index(drop=True)

    final_df.to_csv(CONSOLIDATED_OUTPUT_FILE, sep=";", index=False, encoding="utf-8")

    print(f"File generated: {CONSOLIDATED_OUTPUT_FILE}")
    print(f"Total rows: {len(final_df)}")

# Pipeline execution

In [ ]:
filter_wpi_ports()
filter_cnsp_ports()
consolidate_ports()